# 01. Dataset Verification

Kiểm tra split, schema gốc và schema chuẩn hóa của dataset.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
MAPPINGS_DIR = ROOT / "outputs" / "mappings"
REPORTS_DIR = ROOT / "outputs" / "reports"
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

## 2. Kiểm tra các file đầu ra bắt buộc

In [ ]:
required_files = {
    "raw_train": RAW_DIR / "train.csv",
    "raw_validation": RAW_DIR / "validation.csv",
    "raw_test": RAW_DIR / "test.csv",
    "raw_metadata": RAW_DIR / "dataset_source_metadata.json",
    "processed_train": PROCESSED_DIR / "train_standardized.csv",
    "processed_validation": PROCESSED_DIR / "validation_standardized.csv",
    "processed_test": PROCESSED_DIR / "test_standardized.csv",
    "processed_all": PROCESSED_DIR / "all_standardized.csv",
    "sentiment_mapping": MAPPINGS_DIR / "sentiment_label_mapping.json",
    "topic_mapping": MAPPINGS_DIR / "topic_label_mapping.json",
    "split_summary": TABLES_DIR / "split_summary.csv",
    "quality_summary": TABLES_DIR / "dataset_quality_summary.csv",
    "dataset_schema": REPORTS_DIR / "dataset_schema.json",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("All required Stage 1 files exist.")

## 3. Split summary

Nội dung: xác nhận dataset có đủ `train`, `validation`, `test` và số mẫu từng split đúng với kết quả verification.

In [ ]:
split_summary = pd.read_csv(TABLES_DIR / "split_summary.csv")
display(split_summary)

## 4. Data quality summary

Nội dung: kiểm tra text rỗng, missing label và duplicate text sau khi chuẩn hóa.

In [ ]:
quality_summary = pd.read_csv(TABLES_DIR / "dataset_quality_summary.csv")
display(quality_summary)

## 5. Official label mappings

Mapping phải lấy từ HuggingFace `ClassLabel`, không tự suy đoán.

In [ ]:
with open(MAPPINGS_DIR / "sentiment_label_mapping.json", "r", encoding="utf-8") as f:
    sentiment_mapping = json.load(f)

with open(MAPPINGS_DIR / "topic_label_mapping.json", "r", encoding="utf-8") as f:
    topic_mapping = json.load(f)

display(Markdown("### Sentiment mapping"))
display(sentiment_mapping)

display(Markdown("### Topic mapping"))
display(topic_mapping)

## 6. Dataset schema

Nội dung: xác nhận cột gốc và cột chuẩn hóa dùng cho toàn bộ pipeline sau.

In [ ]:
with open(REPORTS_DIR / "dataset_schema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

display(schema.get("official_schema", schema))

## 7. Kiểm tra dữ liệu chuẩn hóa

Các giai đoạn sau sẽ đọc `data/processed/*_standardized.csv`, không đọc trực tiếp HuggingFace.

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train_standardized.csv")
valid_df = pd.read_csv(PROCESSED_DIR / "validation_standardized.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_standardized.csv")

display(train_df.head())
print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

expected_cols = [
    "id",
    "split",
    "text",
    "sentiment_label",
    "sentiment_name",
    "topic_label",
    "topic_name",
]

missing_cols = [col for col in expected_cols if col not in train_df.columns]
if missing_cols:
    raise ValueError(f"Missing standardized columns: {missing_cols}")

print("Standardized schema is valid.")

## 8. Kết luận Stage 1

Giai đoạn Dataset Verification hoàn thành khi các điểm sau đúng:

```text
- Dataset có đủ 3 split: train, validation, test.
- Cột text gốc là sentence và được chuẩn hóa thành text.
- Sentiment có 3 lớp: negative, neutral, positive.
- Topic có 4 lớp: lecturer, training_program, facility, others.
- Mapping label được lấy từ ClassLabel chính thức.
- Raw snapshot được lưu trong data/raw/.
- Dữ liệu chuẩn hóa được lưu trong data/processed/.
- Các giai đoạn sau dùng schema chuẩn hóa, không dùng trực tiếp schema gốc.
```

Ý nghĩa:

```text
Stage 1 tạo data contract cho toàn bộ dự án.
Nếu không có bước này, các kết quả baseline, PhoBERT và robustness evaluation có thể sai do nhầm split, nhầm label hoặc nhầm schema.
```